### GSE111629

In [8]:
import polars as pl
import gzip
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE111629")

# ── Build barcode → GSM mapping from series matrix ───────────────────────────
gsm_ids, titles = [], []
with gzip.open(RAW / "GSE111629_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_title" in line:
            titles  = [t.replace('"', "").replace("genomic DNA from ", "").strip()
                       for t in line.strip().split("\t")[1:]]

# parquet columns have X prepended by R to numeric names
barcode_to_gsm = {"X" + barcode: gsm for barcode, gsm in zip(titles, gsm_ids)}
print(f"Mapping built: {len(barcode_to_gsm)} entries")
print(f"Example: {list(barcode_to_gsm.items())[:3]}")

# ── Load, rename, set index, save ────────────────────────────────────────────
df = pl.read_parquet(RAW / "GSE111629_PEGblood_450kMethylationDataBackgroundNormalized.parquet")

# Rename SampleID → index, rename barcode columns → GSM IDs
rename_map = {"SampleID": "ProbeID"} | barcode_to_gsm
df = df.rename({k: v for k, v in rename_map.items() if k in df.columns})

# Set ProbeID as index by moving it to first position (polars has no index)
# Save with ProbeID as the first column — the pipeline sets it as index on load
print(f"Shape after rename: {df.shape}")
print(f"First 5 columns: {df.columns[:5]}")

df.write_parquet(RAW / "GSE111629_PEGblood_450kMethylationDataBackgroundNormalized.parquet")
print("Done — parquet overwritten with GSM IDs as column names.")

Mapping built: 572 entries
Example: [('X3999979001_R01C01', 'GSM3035401'), ('X3999979001_R01C02', 'GSM3035402'), ('X3999979001_R02C01', 'GSM3035403')]
Shape after rename: (485512, 574)
First 5 columns: ['ProbeID', 'GSM3035401', 'GSM3035402', 'GSM3035403', 'GSM3035404']
Done — parquet overwritten with GSM IDs as column names.


### GSE84727

In [43]:
import polars as pl
import gzip
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE84727")

# ── Get GSM IDs in order from series matrix ───────────────────────────────────
with gzip.open(RAW / "GSE84727_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_characteristics_ch1" in line:
            col_ids = [i.replace('"', "").replace("sentrixids:","").strip() for i in line.strip().split("\t")[1:]]
            if '3998567027_R01C01' in col_ids :
                break

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")
print(f"COL IDs: {len(col_ids)}, first 3: {col_ids[:3]}")

# ── Load parquet and rename columns positionally ──────────────────────────────
df = pl.read_parquet(RAW / "GSE84727_normalisedBetas.parquet")

internal_to_gsm = dict(zip(col_ids, gsm_ids))
rename_map = {"": "ProbeID"} | {c: internal_to_gsm.get(c, c) for c in df.columns[1:]}
df = df.rename(rename_map)

print(f"First 5 columns after rename: {df.columns[:5]}")
print(f"Shape: {df.shape}")

# ── Overwrite parquet ─────────────────────────────────────────────────────────
df.write_parquet(RAW / "GSE84727_normalisedBetas.parquet")
print("Done — parquet overwritten with GSM IDs as column names.")

GSM IDs: 847, first 3: ['GSM2250273', 'GSM2250274', 'GSM2250275']
COL IDs: 847, first 3: ['3998567027_R01C01', '3998567027_R02C01', '3998567027_R04C01']
First 5 columns after rename: ['ProbeID', 'GSM2250273', 'GSM2250274', 'GSM2250706', 'GSM2250275']
Shape: (420374, 848)
Done — parquet overwritten with GSM IDs as column names.


### GSE55763

In [14]:
import polars as pl
import gzip
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE55763")

# ── Get GSM IDs in order ──────────────────────────────────────────────────────
with gzip.open(RAW / "GSE55763_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_title" in line:
            col_ids = [i.replace('"', "").replace("Peripheral blood,","").strip() for i in line.strip().split("\t")[1:]]

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")
print(f"COL IDs: {len(col_ids)}, first 3: {col_ids[:3]}")

# ── Load parquet and rename columns positionally ──────────────────────────────
df = pl.read_parquet(RAW / "GSE55763_normalized_betas.parquet")

# ── Reorder columns ──────────────────────────────
df = df[['ID_REF'] + col_ids]

# ── Rename barcode columns → GSM IDsb positionally ────────────────────────────
barcode_cols = df.columns[1:]   # everything after ID_REF
print(f"Barcode cols: {len(barcode_cols)}, GSM IDs: {len(gsm_ids)}")

rename_map = {"ID_REF": "ProbeID"} | dict(zip(barcode_cols, gsm_ids))
df = df.rename(rename_map)

print(f"First 5 columns after rename: {df.columns[:5]}")

# ── Overwrite parquet ─────────────────────────────────────────────────────────
df.write_parquet(RAW / "GSE55763_normalized_betas.parquet")
print("Done.")

GSM IDs: 2711, first 3: ['GSM1343050', 'GSM1343051', 'GSM1343052']
COL IDs: 2711, first 3: ['7786915023_R02C02', '7786915135_R04C02', '7471147149_R06C01']
Barcode cols: 2711, GSM IDs: 2711
First 5 columns after rename: ['ProbeID', 'GSM1343050', 'GSM1343051', 'GSM1343052', 'GSM1343053']


### GSE169156

In [39]:
import polars as pl
import gzip
import re
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE169156")

# ── Get GSM IDs and internal IDs from Sample_title ───────────────────────────
with gzip.open(RAW / "GSE169156_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_title" in line:
            internal_ids = [
                re.search(r'\[(.+?)\]', t).group(1)
                for t in line.strip().split("\t")[1:]
            ]

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")
print(f"Internal IDs: {len(internal_ids)}, first 3: {internal_ids[:3]}")

# ── Load, remap, save ─────────────────────────────────────────────────────────
df = pl.read_parquet(RAW / "GSE169156_SJLIFE_IlluminaEPIC_2052samples_GEO_03162021_processed.parquet")

internal_to_gsm = dict(zip(internal_ids, gsm_ids))
rename_map = {"ID_REF": "ProbeID"} | {c: internal_to_gsm.get(c, c) for c in df.columns[1:]}
df = df.rename(rename_map)

print(f"First 5 columns after rename: {df.columns[:5]}")

df.write_parquet(RAW / "GSE169156_SJLIFE_IlluminaEPIC_2052samples_GEO_03162021_processed.parquet")
print("Done.")

GSM IDs: 2052, first 3: ['GSM5177045', 'GSM5177046', 'GSM5177047']
Internal IDs: 2052, first 3: ['SJNHL017924_G1', 'SJSTS018869_G1', 'SJALL018469_G1']
First 5 columns after rename: ['ProbeID', 'GSM5177913', 'GSM5177914', 'GSM5177915', 'GSM5177916']
Done.


### GSE157131

In [39]:
import polars as pl
import gzip
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE157131")

# ── Get GSM IDs in order ──────────────────────────────────────────────────────
with gzip.open(RAW / "GSE157131-GPL21145.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_description" in line:
            col_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]

print(f"GSM IDs: {len(gsm_ids_1)}, first 3: {gsm_ids_1[:3]}")
print(f"COL IDs: {len(col_ids_1)}, first 3: {col_ids_1[:3]}")

with gzip.open(RAW / "GSE157131-GPL13534.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids_2 = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_description" in line:
            col_ids_2 = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]

print(f"GSM IDs: {len(gsm_ids_2)}, first 3: {gsm_ids_2[:3]}")
print(f"COL IDs: {len(col_ids_2)}, first 3: {col_ids_2[:3]}")

#gsm_ids = gsm_ids_1 + gsm_ids_2
#col_ids = col_ids_1 + col_ids_2

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")
print(f"COL IDs: {len(col_ids)}, first 3: {col_ids[:3]}")


# ── Load parquet and rename positionally ─────────────────────────────────────
df = pl.read_parquet(RAW / "GSE157131_Matrix_processed_beta_geo_08252020.parquet")

internal_to_gsm = dict(zip(col_ids, gsm_ids))
rename_map = {"ID_REF": "ProbeID"} | {c: internal_to_gsm.get(c, c) for c in df.columns[1:]}
df = df.rename(rename_map)

barcode_cols = df.columns[1:]  # skip ID_REF
print(f"Barcode cols: {len(barcode_cols)}, GSM IDs: {len(gsm_ids)}")
print(f"First 5 columns after rename: {df.columns[:5]}")

# ── Overwrite parquet ─────────────────────────────────────────────────────────
df.write_parquet(RAW / "GSE157131_Matrix_processed_beta_geo_08252020.parquet")
print("Done.")

GSM IDs: 946, first 3: ['GSM4753639', 'GSM4753640', 'GSM4753641']
COL IDs: 946, first 3: ['AAGENOA_LCL_0001', 'AAGENOA_LCL_0002', 'AAGENOA_LCL_0003']
GSM IDs: 272, first 3: ['GSM4753643', 'GSM4753647', 'GSM4753648']
COL IDs: 272, first 3: ['AAGENOA_LCL_0006', 'AAGENOA_LCL_0010', 'AAGENOA_LCL_0011']
GSM IDs: 946, first 3: ['GSM4753639', 'GSM4753640', 'GSM4753641']
COL IDs: 946, first 3: ['AAGENOA_LCL_0001', 'AAGENOA_LCL_0002', 'AAGENOA_LCL_0003']
Barcode cols: 1218, GSM IDs: 946
First 5 columns after rename: ['ProbeID', 'GSM4753639', 'GSM4753640', 'GSM4753641', 'GSM4753642']
Done.


In [22]:
df

ProbeID,GSM4753639,GSM4753640,GSM4753641,GSM4753642,GSM4753643,GSM4753644,GSM4753645,GSM4753646,GSM4753647,GSM4753648,GSM4753649,GSM4753650,GSM4753651,GSM4753652,GSM4753653,GSM4753654,GSM4753655,GSM4753656,GSM4753657,GSM4753658,GSM4753659,GSM4753660,GSM4753661,GSM4753662,GSM4753663,GSM4753664,GSM4753665,GSM4753666,GSM4753667,GSM4753668,GSM4753669,GSM4753670,GSM4753671,GSM4753672,GSM4753673,GSM4753674,…,GSM4754820,GSM4754821,GSM4754822,GSM4754823,GSM4754824,GSM4754825,GSM4754826,GSM4754827,GSM4754828,GSM4754829,GSM4754830,GSM4754831,GSM4754832,GSM4754833,GSM4754834,GSM4754835,GSM4754836,GSM4754837,GSM4754838,GSM4754839,GSM4754840,GSM4754841,GSM4754842,GSM4754843,GSM4754844,GSM4754845,GSM4754846,GSM4754847,GSM4754848,GSM4754849,GSM4754850,GSM4754851,GSM4754852,GSM4754853,GSM4754854,GSM4754855,GSM4754856
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""cg00000029""",0.743143,0.706606,0.628885,0.562915,0.614113,0.61417,0.621852,0.702323,0.670715,0.602085,0.637377,0.810542,0.524458,0.705846,0.704709,0.637979,0.765041,0.752058,0.698944,0.74734,0.594071,0.672425,0.714021,0.632401,0.599509,0.604297,0.669404,0.759007,0.66768,0.639474,0.725226,0.663143,0.644101,0.744332,0.51345,0.598168,…,0.572035,0.597037,0.754375,0.705512,0.700928,0.741055,0.696354,0.63886,0.596386,0.665725,0.721606,0.62104,0.701058,0.608731,0.742045,0.686269,0.752356,0.689075,0.558517,0.616811,0.690211,0.730824,0.687066,0.544552,0.624664,0.593885,0.695128,0.655178,0.595505,0.624226,0.604256,0.616109,0.613575,0.652918,0.606318,0.501986,0.698725
"""cg00000109""",0.956468,0.953614,0.962331,0.952262,0.94954,0.952814,0.966259,0.955471,0.962871,0.938608,0.951186,0.955195,0.953014,0.94309,0.956676,0.949236,0.948712,0.957397,0.957561,0.949693,0.945948,0.96353,0.95643,0.957817,0.951135,0.957808,0.957935,0.95176,0.949498,0.966643,0.957436,0.965143,0.959079,0.963155,0.954505,0.957223,…,0.949419,0.958058,0.950554,0.961506,0.961586,0.949962,0.952241,0.947667,0.952464,0.953383,0.956192,0.946605,0.960191,0.9548,0.955375,0.950114,0.963771,0.948481,0.957596,0.955248,0.966078,0.958822,0.949273,0.964595,0.955929,0.956543,0.957178,0.948953,0.963924,0.952129,0.949459,0.95229,0.94644,0.967401,0.953985,0.942613,0.960438
"""cg00000165""",0.151411,0.127816,0.092166,0.077027,0.158545,0.120263,0.106238,0.122761,0.092052,0.135179,0.081362,0.099472,0.11249,0.122408,0.139771,0.07958,0.073689,0.119332,0.203557,0.140518,0.108219,0.103103,0.135681,0.098683,0.160858,0.211254,0.101864,0.16377,0.123972,0.166769,0.131638,0.127294,0.186685,0.110509,0.140292,0.108948,…,0.117532,0.125535,0.185964,0.149593,0.102747,0.188398,0.199959,0.135354,0.10216,0.216493,0.156528,0.128493,0.167962,0.170633,0.102047,0.068985,0.109873,0.089624,0.11763,0.105932,0.11241,0.183197,0.179602,0.13236,0.1264,0.188597,0.155702,0.098844,0.174793,0.202968,0.195232,0.112815,0.116362,0.105521,0.078599,0.117287,0.138814
"""cg00000236""",0.90472,0.874442,0.863355,0.894204,0.88636,0.876077,0.896574,0.939659,0.888558,0.898843,0.939136,0.937654,0.863238,0.829431,0.917789,0.879665,0.915455,0.925571,0.88948,0.941864,0.905829,0.882314,0.903327,0.903889,0.934421,0.913471,0.911507,0.929512,0.840457,0.845937,0.909752,0.833018,0.887208,0.894814,0.853017,0.850993,…,0.916491,0.89796,0.905619,0.880321,0.908299,0.910654,0.860174,0.919424,0.874917,0.893894,0.922365,0.903632,0.890328,0.916683,0.87767,0.906361,0.927442,0.896267,0.91114,0.898022,0.887359,0.922718,0.917311,0.927855,0.893444,0.851333,0.872895,0.909783,0.895368,0.859739,0.877622,0.893032,0.87652,0.933095,0.889046,0.912986,0.927887
"""cg00000289""",0.864994,0.845839,0.883417,0.891442,0.867769,0.873706,0.878726,0.854738,0.895476,0.819145,0.845482,0.874773,0.848737,0.864937,0.863906,0.867059,0.855363,0.866727,0.877467,0.875899,0.875303,0.895066,0

### GSE210255

In [37]:
import polars as pl
import gzip
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE210255")

# ── Get GSM IDs in order ──────────────────────────────────────────────────────
with gzip.open(RAW / "GSE210255_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")

# ── Load parquet and rename positionally ─────────────────────────────────────
df = pl.read_parquet(RAW / "GSE210255_Beta_value_EPIC.parquet")

barcode_cols = df.columns[1:]  # skip probe ID column
print(f"Sample cols: {len(barcode_cols)}, GSM IDs: {len(gsm_ids)}")

rename_map = {df.columns[0]: "ProbeID"} | dict(zip(barcode_cols, gsm_ids))
df = df.rename(rename_map)

print(f"First 5 columns after rename: {df.columns[:5]}")

df.write_parquet(RAW / "GSE210255_Beta_value_EPIC.parquet")
print("Done.")

GSM IDs: 1394, first 3: ['GSM6425154', 'GSM6425155', 'GSM6425156']
Sample cols: 1394, GSM IDs: 1394
First 5 columns after rename: ['ProbeID', 'GSM6425154', 'GSM6425155', 'GSM6425156', 'GSM6425157']
Done.


### GSE179325

In [42]:
import polars as pl
import gzip
import re
from pathlib import Path

RAW = Path("/work/gr-fe/bryan/EpiHydra/data/01_raw/GSE179325")

# ── Get GSM IDs and internal IDs from Sample_title ───────────────────────────
with gzip.open(RAW / "GSE179325_series_matrix.txt.gz", "rt") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if "Sample_geo_accession" in line:
            gsm_ids = [i.replace('"', "").strip() for i in line.strip().split("\t")[1:]]
        if "Sample_title" in line:
            internal_ids = [
                re.search(r'\[(.+?)\]', t).group(1)
                for t in line.strip().split("\t")[1:]
            ]

print(f"GSM IDs: {len(gsm_ids)}, first 3: {gsm_ids[:3]}")
print(f"Internal IDs: {len(internal_ids)}, first 3: {internal_ids[:3]}")

# ── Load, remap, save ─────────────────────────────────────────────────────────
df = pl.read_parquet(RAW / "GSE179325_B.Methylation.parquet")

internal_to_gsm = dict(zip(internal_ids, gsm_ids))
rename_map = {"ID": "ProbeID"} | {c: internal_to_gsm.get(c, c) for c in df.columns[1:]}
df = df.rename(rename_map)

print(f"First 5 columns after rename: {df.columns[:5]}")

df.write_parquet(RAW / "GSE179325_B.Methylation.parquet")
print("Done.")

GSM IDs: 574, first 3: ['GSM5414278', 'GSM5414279', 'GSM5414280']
Internal IDs: 574, first 3: ['100120', '100165', '101417']
First 5 columns after rename: ['ProbeID', 'GSM5414278', 'GSM5414279', 'GSM5414280', 'GSM5414281']
Done.
